# Publish RAF-DB 7 Emotions MediaPipe 768 Dataset

Self-contained notebook for a remote server.

It downloads `rhavill/raf-db-7emotions` from Hugging Face, creates stable labels and stratified `train/val/test` splits, extracts MediaPipe face landmarks at 768 x 768 processing resolution, and pushes the resulting dataset to Hugging Face as `raf-db-7emotions-mediapipe-768`.

No local project files are required.


In [2]:
# Run this first on the remote server.
import subprocess
import sys

INSTALL_PACKAGES = True

PACKAGES = [
    'numpy==1.23.5',
    'pandas==2.2.3',
    'scikit-learn==1.5.2',
    'datasets==4.8.5',
    'huggingface-hub==1.16.1',
    'hf-transfer==0.1.8',
    'pyarrow==21.0.0',
    'pillow==10.4.0',
    'opencv-python-headless==4.10.0.84',
    'mediapipe==0.10.18',
    'protobuf==4.25.5',
    'tqdm==4.66.5',
    'python-dotenv==1.0.1',
]

if INSTALL_PACKAGES:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', *PACKAGES])


  Using cached numpy-1.23.5-cp310-cp310-macosx_11_0_arm64.whl.metadata (2.3 kB)
  Using cached datasets-4.8.5-py3-none-any.whl.metadata (19 kB)
  Using cached huggingface_hub-1.16.1-py3-none-any.whl.metadata (14 kB)
  Using cached mediapipe-0.10.18-cp310-cp310-macosx_11_0_universal2.whl.metadata (9.7 kB)
  Using cached python_dotenv-1.0.1-py3-none-any.whl.metadata (23 kB)
Using cached numpy-1.23.5-cp310-cp310-macosx_11_0_arm64.whl (13.4 MB)
Using cached datasets-4.8.5-py3-none-any.whl (528 kB)
Using cached huggingface_hub-1.16.1-py3-none-any.whl (668 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.2/31.2 MB 52.2 MB/s  0:00:00m0:00:0100:01
Using cached mediapipe-0.10.18-cp310-cp310-macosx_11_0_universal2.whl (49.1 MB)
Using cached python_dotenv-1.0.1-py3-none-any.whl (19 kB)
  Attempting uninstall: python-dotenv
    Found existing installation: python-dotenv 1.2.2
    Uninstalling python-dotenv-1.2.2:
      Successfully uninstalled python-dotenv-1.2.2
  Attempting uninstall: pyarrow


In [2]:
!pip3 show mediapipe

zsh:1: /Users/pelmeshek1706/Desktop/projects/airest-face/.venv/bin/pip3: bad interpreter: /Users/pelmeshek1706/Desktop/projects/airest-voice/.venv/bin/python: no such file or directory


## Imports And Configuration

No token is required to download the public source dataset. If `HF_TOKEN` is stored in `.env`, the config cell below loads it automatically for the final push step.


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Optional
import json
import os
import random
import shutil
import urllib.request

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from datasets import ClassLabel, Dataset, DatasetDict, Features, Image as HFImage, Sequence, Value, load_dataset
from dotenv import load_dotenv
from huggingface_hub import HfApi, get_token, notebook_login
from PIL import Image as PILImage
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

load_dotenv()

SOURCE_DATASET_ID = 'rhavill/raf-db-7emotions'
TARGET_DATASET_NAME = 'raf-db-7emotions-mediapipe-768'
TARGET_REPO_ID = TARGET_DATASET_NAME  # if no namespace is provided, the authenticated user's namespace is used
PRIVATE_REPO = False

RUN_DIR = Path('rafdb_mediapipe_768_publish')
CACHE_DIR = RUN_DIR / 'cache'
HF_CACHE_DIR = CACHE_DIR / 'hf'
SOURCE_CACHE_DIR = HF_CACHE_DIR / 'source_dataset'
EXPORT_DIR = RUN_DIR / 'export'

SEED = 42
IMAGE_SIZE = 768
TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15
MAX_SAMPLES = None  # keep None for full dataset; use an integer only for a dry run
REUSE_PROCESSED_CACHE = True
PUSH_TO_HUB = True

TARGET_LABELS = ['anger', 'disgust', 'fear', 'happiness', 'sadness', 'surprise', 'neutral']
TARGET_LABEL_TO_ID = {name: idx for idx, name in enumerate(TARGET_LABELS)}
ID_TO_TARGET_LABEL = {idx: name for name, idx in TARGET_LABEL_TO_ID.items()}
LANDMARK_COUNT = 478
LEFT_EYE_CORNERS = (33, 133)
RIGHT_EYE_CORNERS = (362, 263)

for path in (CACHE_DIR, HF_CACHE_DIR, SOURCE_CACHE_DIR, EXPORT_DIR):
    path.mkdir(parents=True, exist_ok=True)

os.environ.setdefault('HF_HOME', str(HF_CACHE_DIR.resolve()))
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)

set_seed(SEED)

def reset_source_cache() -> None:
    if SOURCE_CACHE_DIR.exists():
        shutil.rmtree(SOURCE_CACHE_DIR)
    SOURCE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def load_source_dataset(dataset_id: str, split: str = 'train'):
    last_error = None
    for attempt_index, download_mode in enumerate(('reuse_dataset_if_exists', 'force_redownload'), start=1):
        try:
            dataset = load_dataset(
                dataset_id,
                split=split,
                cache_dir=str(SOURCE_CACHE_DIR),
                download_mode=download_mode,
            )
            print(f'loaded {dataset_id} with download_mode={download_mode}')
            return dataset
        except Exception as exc:
            last_error = exc
            print(f'load attempt {attempt_index} failed: {type(exc).__name__}: {exc}')
            reset_source_cache()
    raise RuntimeError(f'Failed to load {dataset_id} after cache reset and redownload') from last_error

print('target dataset:', TARGET_REPO_ID)
print('run dir:', RUN_DIR.resolve())

/Users/pelmeshek1706/Desktop/projects/airest-face/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


target dataset: raf-db-7emotions-mediapipe-768
run dir: /Users/pelmeshek1706/Desktop/projects/airest-face/output/jupyter-notebook/rafdb_mediapipe_768_publish


## Download Source Dataset And Build Labels

The source dataset contains one HF split named `train`. This notebook creates new stratified `train`, `val`, and `test` splits using the same ratio as the training notebook: 70/15/15 with `SEED=42`.


In [4]:
source_ds = load_source_dataset(SOURCE_DATASET_ID, split='train')
if MAX_SAMPLES is not None:
    source_ds = source_ds.select(range(min(int(MAX_SAMPLES), len(source_ds))))

hf_label_names = list(source_ds.features['label'].names)
unknown_labels = sorted(set(hf_label_names) - set(TARGET_LABEL_TO_ID))
if unknown_labels:
    raise ValueError(f'Unexpected source labels: {unknown_labels}')

hf_to_target_id = {idx: TARGET_LABEL_TO_ID[name] for idx, name in enumerate(hf_label_names)}
source_labels = np.asarray(source_ds['label'], dtype=np.int64)
target_labels = np.asarray([hf_to_target_id[int(label)] for label in source_labels], dtype=np.int64)

label_map_df = pd.DataFrame([
    {
        'hf_label_id': idx,
        'hf_label_name': name,
        'label': hf_to_target_id[idx],
        'label_name': ID_TO_TARGET_LABEL[hf_to_target_id[idx]],
    }
    for idx, name in enumerate(hf_label_names)
])

print(source_ds)
display(label_map_df)
display(pd.Series([ID_TO_TARGET_LABEL[int(x)] for x in target_labels]).value_counts().rename_axis('label_name').reset_index(name='count'))


Generating train split: 100%|██████████| 20471/20471 [00:01<00:00, 17566.14 examples/s] 


loaded rhavill/raf-db-7emotions with download_mode=reuse_dataset_if_exists
Dataset({
    features: ['image', 'bbox', 'label'],
    num_rows: 20471
})


,hf_label_id,hf_label_name,label,label_name
0,0,anger,0,anger
1,1,disgust,1,disgust
2,2,fear,2,fear
3,3,happiness,3,happiness
4,4,neutral,6,neutral
5,5,sadness,4,sadness
6,6,surprise,5,surprise


,label_name,count
0,happiness,5957
1,neutral,5132
2,anger,4071
3,sadness,2460
4,surprise,1619
5,disgust,877
6,fear,355


In [5]:
if not np.isclose(TRAIN_SIZE + VAL_SIZE + TEST_SIZE, 1.0):
    raise ValueError('TRAIN_SIZE + VAL_SIZE + TEST_SIZE must equal 1.0')

all_indices = np.arange(len(source_ds), dtype=np.int64)
train_idx, temp_idx = train_test_split(
    all_indices,
    test_size=(VAL_SIZE + TEST_SIZE),
    random_state=SEED,
    stratify=target_labels,
)
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=TEST_SIZE / (VAL_SIZE + TEST_SIZE),
    random_state=SEED,
    stratify=target_labels[temp_idx],
)

split_indices = {
    'train': np.sort(train_idx),
    'val': np.sort(val_idx),
    'test': np.sort(test_idx),
}

split_preview = []
for split_name, indices in split_indices.items():
    for label_id in range(len(TARGET_LABELS)):
        split_preview.append({
            'split': split_name,
            'label_name': ID_TO_TARGET_LABEL[label_id],
            'count': int((target_labels[indices] == label_id).sum()),
        })

split_preview_df = pd.DataFrame(split_preview).pivot(index='split', columns='label_name', values='count').fillna(0).astype(int)
display(split_preview_df[TARGET_LABELS])
print({name: int(len(indices)) for name, indices in split_indices.items()})


label_name,anger,disgust,fear,happiness,sadness,surprise,neutral
split,,,,,,,
test,611,131,54,893,369,243,770
train,2850,614,248,4170,1722,1133,3592
val,610,132,53,894,369,243,770


{'train': 14329, 'val': 3071, 'test': 3071}


## MediaPipe 768 Landmark Extraction

Each row keeps the original source image and adds MediaPipe landmark columns extracted after resizing the image to 768 x 768.

Rows where MediaPipe fails are still kept in the same split with `landmark_success=False` and empty landmark arrays.


In [6]:
FACE_LANDMARKER_URL = 'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task'
FACE_LANDMARKER_PATH = CACHE_DIR / 'face_landmarker.task'

def ensure_face_landmarker_model() -> Path:
    if not FACE_LANDMARKER_PATH.exists():
        urllib.request.urlretrieve(FACE_LANDMARKER_URL, FACE_LANDMARKER_PATH)
    return FACE_LANDMARKER_PATH

def build_face_landmarker():
    model_path = ensure_face_landmarker_model()
    BaseOptions = mp.tasks.BaseOptions
    FaceLandmarker = mp.tasks.vision.FaceLandmarker
    FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
    VisionRunningMode = mp.tasks.vision.RunningMode
    options = FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=str(model_path)),
        running_mode=VisionRunningMode.IMAGE,
        num_faces=1,
        output_face_blendshapes=False,
        output_facial_transformation_matrixes=True,
    )
    return FaceLandmarker.create_from_options(options)

def pil_rgb(image: Any) -> PILImage.Image:
    if isinstance(image, PILImage.Image):
        return image.convert('RGB')
    return PILImage.fromarray(np.asarray(image, dtype=np.uint8)).convert('RGB')

def resize_for_mediapipe(image: Any, image_size: int = IMAGE_SIZE) -> np.ndarray:
    pil = pil_rgb(image)
    arr = np.asarray(pil, dtype=np.uint8)
    if arr.shape[:2] != (image_size, image_size):
        arr = cv2.resize(arr, (image_size, image_size), interpolation=cv2.INTER_CUBIC)
    return arr

def normalize_landmarks(raw: np.ndarray) -> tuple[np.ndarray, float, float]:
    raw = np.asarray(raw, dtype=np.float32)
    if raw.shape != (LANDMARK_COUNT, 3):
        raise ValueError(f'Expected {(LANDMARK_COUNT, 3)}, got {raw.shape}')
    if not np.isfinite(raw).all():
        raise ValueError('Non-finite landmark coordinates')

    left_eye = raw[list(LEFT_EYE_CORNERS), :2].mean(axis=0)
    right_eye = raw[list(RIGHT_EYE_CORNERS), :2].mean(axis=0)
    center_xy = (left_eye + right_eye) / 2.0
    eye_vec = right_eye - left_eye
    scale = float(np.linalg.norm(eye_vec))
    if scale < 1e-6 or not np.isfinite(scale):
        raise ValueError('Invalid inter-eye scale')

    roll = float(np.arctan2(eye_vec[1], eye_vec[0]))
    cos_a = float(np.cos(-roll))
    sin_a = float(np.sin(-roll))
    rotation = np.array([[cos_a, -sin_a], [sin_a, cos_a]], dtype=np.float32)

    out = raw.copy()
    out[:, :2] = (out[:, :2] - center_xy) @ rotation.T
    out[:, :2] /= scale
    z_center = float(raw[list(LEFT_EYE_CORNERS + RIGHT_EYE_CORNERS), 2].mean())
    out[:, 2] = (out[:, 2] - z_center) / scale
    return out.astype(np.float32), scale, roll

def empty_landmarks() -> list[list[float]]:
    return []

def process_row(source_index: int, split_name: str, landmarker: Any) -> dict[str, Any]:
    row = source_ds[int(source_index)]
    hf_label_id = int(row['label'])
    label_id = int(hf_to_target_id[hf_label_id])
    label_name = ID_TO_TARGET_LABEL[label_id]
    image = pil_rgb(row['image'])
    sample_id = f'{split_name}_{source_index:06d}'

    base = {
        'image': image,
        'sample_id': sample_id,
        'source_dataset': SOURCE_DATASET_ID,
        'source_index': int(source_index),
        'split': split_name,
        'hf_label_id': hf_label_id,
        'hf_label_name': hf_label_names[hf_label_id],
        'label': label_id,
        'label_name': label_name,
        'mediapipe_image_size': IMAGE_SIZE,
    }

    try:
        image_768 = resize_for_mediapipe(image, IMAGE_SIZE)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_768)
        result = landmarker.detect(mp_image)
        if not result.face_landmarks:
            raise RuntimeError('no_face_detected')

        raw = np.asarray([[lm.x, lm.y, lm.z] for lm in result.face_landmarks[0]], dtype=np.float32)
        norm, scale, roll = normalize_landmarks(raw)
        pixel = raw.copy()
        pixel[:, 0] *= IMAGE_SIZE
        pixel[:, 1] *= IMAGE_SIZE
        pixel[:, 2] *= IMAGE_SIZE
        bbox = [
            float(np.min(pixel[:, 0])),
            float(np.min(pixel[:, 1])),
            float(np.max(pixel[:, 0])),
            float(np.max(pixel[:, 1])),
        ]
        transform = []
        if getattr(result, 'facial_transformation_matrixes', None):
            transform = np.asarray(result.facial_transformation_matrixes[0], dtype=np.float32).reshape(-1).tolist()

        return {
            **base,
            'landmark_success': True,
            'failure_reason': '',
            'landmarks_mediapipe_xyz': raw.tolist(),
            'landmarks_pixel_xyz_768': pixel.astype(np.float32).tolist(),
            'landmarks_stable_eye_norm': norm.tolist(),
            'bbox_xyxy_768': bbox,
            'inter_eye_scale': float(scale),
            'roll_radians': float(roll),
            'facial_transformation_matrix': transform,
        }
    except Exception as exc:
        return {
            **base,
            'landmark_success': False,
            'failure_reason': str(exc),
            'landmarks_mediapipe_xyz': empty_landmarks(),
            'landmarks_pixel_xyz_768': empty_landmarks(),
            'landmarks_stable_eye_norm': empty_landmarks(),
            'bbox_xyxy_768': [],
            'inter_eye_scale': None,
            'roll_radians': None,
            'facial_transformation_matrix': [],
        }


In [7]:
features = Features({
    'image': HFImage(),
    'sample_id': Value('string'),
    'source_dataset': Value('string'),
    'source_index': Value('int32'),
    'split': Value('string'),
    'hf_label_id': Value('int32'),
    'hf_label_name': Value('string'),
    'label': ClassLabel(names=TARGET_LABELS),
    'label_name': Value('string'),
    'mediapipe_image_size': Value('int32'),
    'landmark_success': Value('bool'),
    'failure_reason': Value('string'),
    'landmarks_mediapipe_xyz': Sequence(Sequence(Value('float32'))),
    'landmarks_pixel_xyz_768': Sequence(Sequence(Value('float32'))),
    'landmarks_stable_eye_norm': Sequence(Sequence(Value('float32'))),
    'bbox_xyxy_768': Sequence(Value('float32')),
    'inter_eye_scale': Value('float32'),
    'roll_radians': Value('float32'),
    'facial_transformation_matrix': Sequence(Value('float32')),
})

cache_tag = 'full' if MAX_SAMPLES is None else f'max_{MAX_SAMPLES}'
processed_cache_path = CACHE_DIR / f'processed_rows_{cache_tag}.jsonl'

if REUSE_PROCESSED_CACHE and processed_cache_path.exists():
    rows_by_split = {'train': [], 'val': [], 'test': []}
    with processed_cache_path.open('r', encoding='utf-8') as handle:
        for line in handle:
            record = json.loads(line)
            image_path = record.pop('_image_path')
            record['image'] = PILImage.open(image_path).convert('RGB')
            rows_by_split[record['split']].append(record)
else:
    rows_by_split = {'train': [], 'val': [], 'test': []}
    image_cache_dir = CACHE_DIR / f'images_{cache_tag}'
    image_cache_dir.mkdir(parents=True, exist_ok=True)
    landmarker = build_face_landmarker()
    with processed_cache_path.open('w', encoding='utf-8') as handle:
        for split_name, indices in split_indices.items():
            for source_index in tqdm(indices.tolist(), desc=f'process {split_name}'):
                record = process_row(int(source_index), split_name, landmarker)
                image_path = image_cache_dir / f'{record["sample_id"]}.jpg'
                record['image'].save(image_path, format='JPEG', quality=95)
                serializable = dict(record)
                serializable.pop('image')
                serializable['_image_path'] = str(image_path)
                handle.write(json.dumps(serializable) + '\n')
                rows_by_split[split_name].append(record)

dataset_dict = DatasetDict({
    split_name: Dataset.from_list(rows, features=features)
    for split_name, rows in rows_by_split.items()
})

print(dataset_dict)
for split_name, split_ds in dataset_dict.items():
    success_rate = float(np.mean(split_ds['landmark_success'])) if len(split_ds) else 0.0
    print(split_name, 'rows=', len(split_ds), 'landmark_success_rate=', round(success_rate, 4))


I0000 00:00:1779629626.888809 26977373 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M3 Pro
W0000 00:00:1779629626.889958 26977373 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1779629626.894042 27029769 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1779629626.898574 27029777 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
process test: 100%|██████████| 3071/3071 [00:38<00:00, 80.52it/s] 


DatasetDict({
    train: Dataset({
        features: ['image', 'sample_id', 'source_dataset', 'source_index', 'split', 'hf_label_id', 'hf_label_name', 'label', 'label_name', 'mediapipe_image_size', 'landmark_success', 'failure_reason', 'landmarks_mediapipe_xyz', 'landmarks_pixel_xyz_768', 'landmarks_stable_eye_norm', 'bbox_xyxy_768', 'inter_eye_scale', 'roll_radians', 'facial_transformation_matrix'],
        num_rows: 14329
    })
    val: Dataset({
        features: ['image', 'sample_id', 'source_dataset', 'source_index', 'split', 'hf_label_id', 'hf_label_name', 'label', 'label_name', 'mediapipe_image_size', 'landmark_success', 'failure_reason', 'landmarks_mediapipe_xyz', 'landmarks_pixel_xyz_768', 'landmarks_stable_eye_norm', 'bbox_xyxy_768', 'inter_eye_scale', 'roll_radians', 'facial_transformation_matrix'],
        num_rows: 3071
    })
    test: Dataset({
        features: ['image', 'sample_id', 'source_dataset', 'source_index', 'split', 'hf_label_id', 'hf_label_name', 'label', 'l

## Dataset Card


In [8]:
readme_text = f'''---
license: other
task_categories:
- image-classification
- tabular-classification
language:
- en
tags:
- facial-expression-recognition
- raf-db
- mediapipe
- face-landmarks
- emotion-recognition
pretty_name: RAF-DB 7 Emotions MediaPipe 768
size_categories:
- 10K<n<100K
source_datasets:
- {SOURCE_DATASET_ID}
---

# RAF-DB 7 Emotions MediaPipe 768

This dataset is a processed derivative of `{SOURCE_DATASET_ID}` for landmark-based facial expression recognition experiments.

It keeps the source images and labels, remaps labels into a stable seven-class order, creates stratified `train`, `val`, and `test` splits with seed `{SEED}`, and adds MediaPipe Face Landmarker outputs extracted after resizing each image to `{IMAGE_SIZE} x {IMAGE_SIZE}` for processing.

## Labels

Target label order:

```text
{TARGET_LABELS}
```

The `label` column is a `ClassLabel` using that order. The `label_name` column stores the readable class name.

## Splits

The source Hugging Face mirror has a single split. This dataset creates deterministic stratified splits:

- train: {TRAIN_SIZE:.0%}
- val: {VAL_SIZE:.0%}
- test: {TEST_SIZE:.0%}
- seed: `{SEED}`

## Columns

- `image`: source image
- `sample_id`: deterministic row id
- `source_dataset`: original dataset id
- `source_index`: row index in the source dataset
- `split`: split name
- `hf_label_id`, `hf_label_name`: original Hugging Face mirror label
- `label`, `label_name`: remapped target label
- `mediapipe_image_size`: processing size, fixed at `{IMAGE_SIZE}`
- `landmark_success`: whether MediaPipe produced landmarks
- `failure_reason`: extraction failure reason if any
- `landmarks_mediapipe_xyz`: 478 normalized MediaPipe xyz landmarks
- `landmarks_pixel_xyz_768`: 478 xyz landmarks scaled to the `{IMAGE_SIZE} x {IMAGE_SIZE}` processing frame
- `landmarks_stable_eye_norm`: 478 xyz landmarks normalized by stable eye-corner center, scale, and roll
- `bbox_xyxy_768`: landmark-derived face bounding box in the processing frame
- `inter_eye_scale`: eye-corner scale used for normalization
- `roll_radians`: face roll angle used for normalization
- `facial_transformation_matrix`: flattened MediaPipe facial transformation matrix when available

## Intended Use

This dataset is intended for privacy-conscious facial expression recognition experiments where models consume landmark-derived geometry rather than raw texture-heavy image features.

## Caveats

This is a processed Hugging Face mirror dataset. Results should be reported as mirror/proxy results and should include the split seed, MediaPipe processing size, and landmark coverage.

MediaPipe extraction can fail and may fail non-uniformly across classes. Use `landmark_success` and `failure_reason` when training or reporting coverage-adjusted metrics.
'''

readme_path = EXPORT_DIR / 'README.md'
readme_path.write_text(readme_text, encoding='utf-8')
print(readme_path)
print(readme_text[:1200])


rafdb_mediapipe_768_publish/export/README.md
---
license: other
task_categories:
- image-classification
- tabular-classification
language:
- en
tags:
- facial-expression-recognition
- raf-db
- mediapipe
- face-landmarks
- emotion-recognition
pretty_name: RAF-DB 7 Emotions MediaPipe 768
size_categories:
- 10K<n<100K
source_datasets:
- rhavill/raf-db-7emotions
---

# RAF-DB 7 Emotions MediaPipe 768

This dataset is a processed derivative of `rhavill/raf-db-7emotions` for landmark-based facial expression recognition experiments.

It keeps the source images and labels, remaps labels into a stable seven-class order, creates stratified `train`, `val`, and `test` splits with seed `42`, and adds MediaPipe Face Landmarker outputs extracted after resizing each image to `768 x 768` for processing.

## Labels

Target label order:

```text
['anger', 'disgust', 'fear', 'happiness', 'sadness', 'surprise', 'neutral']
```

The `label` column is a `ClassLabel` using that order. The `label_name` column s

## Push To Hugging Face

Set `HF_TOKEN` only if you want non-interactive upload. Otherwise, the notebook will prompt for login before `push_to_hub`.


In [9]:
if PUSH_TO_HUB:
    hf_token = os.environ.get('HF_TOKEN') or get_token()
    if hf_token is None:
        notebook_login()
        hf_token = get_token()
    if hf_token is None:
        raise RuntimeError('HF_TOKEN is required to push the dataset')

    api = HfApi(token=hf_token)
    whoami = api.whoami(token=hf_token)
    namespace = whoami.get('name')
    target_repo_id = TARGET_REPO_ID if '/' in TARGET_REPO_ID else f'{namespace}/{TARGET_REPO_ID}'

    api.create_repo(
        repo_id=target_repo_id,
        repo_type='dataset',
        private=PRIVATE_REPO,
        exist_ok=True,
        token=hf_token,
    )

    dataset_dict.push_to_hub(
        target_repo_id,
        private=PRIVATE_REPO,
        token=hf_token,
        commit_message='Upload RAF-DB 7 emotions MediaPipe 768 dataset',
    )

    api.upload_file(
        path_or_fileobj=str(readme_path),
        path_in_repo='README.md',
        repo_id=target_repo_id,
        repo_type='dataset',
        token=hf_token,
        commit_message='Add dataset card',
    )

    print('Pushed dataset to:', f'https://huggingface.co/datasets/{target_repo_id}')
else:
    print('PUSH_TO_HUB is False; dataset was prepared locally only.')


Creating parquet from Arrow format: 100%|██████████| 5/5 [00:00<00:00, 19.69ba/s]
Processing Files (1 / 1): 100%|██████████|  110MB /  110MB, 8.34MB/s  
New Data Upload: 100%|██████████|  110MB /  110MB, 8.34MB/s  
Creating parquet from Arrow format: 100%|██████████| 5/5 [00:00<00:00, 20.58ba/s]
Processing Files (1 / 1): 100%|██████████|  112MB /  112MB, 8.31MB/s  
New Data Upload: 100%|██████████|  112MB /  112MB, 8.31MB/s  
Creating parquet from Arrow format: 100%|██████████| 5/5 [00:00<00:00, 20.69ba/s]
Processing Files (1 / 1): 100%|██████████|  112MB /  112MB, 8.81MB/s  
New Data Upload: 100%|██████████|  112MB /  112MB, 8.81MB/s  
Creating parquet from Arrow format: 100%|██████████| 5/5 [00:01<00:00,  2.51ba/s]
Processing Files (1 / 1): 100%|██████████| 1.57GB / 1.57GB, 36.1MB/s  
New Data Upload: 100%|██████████| 1.40GB / 1.40GB, 34.0MB/s  
Uploading the dataset shards: 100%|██████████| 4/4 [00:56<00:00, 14.18s/ shards]
Setting num_proc from 1 back to 1 for the val split to disa

Pushed dataset to: https://huggingface.co/datasets/Pelmeshek/raf-db-7emotions-mediapipe-768


In [1]:
!pip show mediapipe

zsh:1: /Users/pelmeshek1706/Desktop/projects/airest-face/.venv/bin/pip: bad interpreter: /Users/pelmeshek1706/Desktop/projects/airest-voice/.venv/bin/python: no such file or directory


## Local Outputs

The notebook writes local cache and export files under `rafdb_mediapipe_768_publish/`. The main artifact is the pushed Hugging Face dataset repository named `raf-db-7emotions-mediapipe-768` under the authenticated account namespace.
